# ForgeGuard: Comparative Evaluation of CNN Architectures in Detecting Digital Receipt Forgery
### Notre Dame of Midsayap College (NDMC) — BSCS Thesis Research
**Researchers**: Rogie P. Bacanto, Daniela S. Ungab  
**Adviser**: Ms. Doris Ann Mariano  

This notebook trains and benchmarks three Convolutional Neural Network (CNN) architectures on **1,003 labeled mobile wallet transaction receipts** (GCash & Maya) preprocessed using **Error Level Analysis (ELA 90Q / 15x)**.

In [ ]:
# Step 1: Clone the thesis repository containing dataset & preprocessors
!git clone https://github.com/DeathKnell837/NDMC-BSCS-THESIS-PREP.git
%cd NDMC-BSCS-THESIS-PREP/thesis-system
!pip install -q Pillow numpy scipy scikit-learn matplotlib seaborn

In [ ]:
# Step 2: Import libraries and verify GPU
import os, glob, time, json
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, applications
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Step 3: Load and Preprocess all 1,003 Dataset Samples with ELA
from preprocessing.ela import compute_ela

IMG_SIZE = (128, 128)
IMAGE_EXTENSIONS = ('*.jpg', '*.jpeg', '*.png', '*.webp')

auth_dir = 'dataset/authentic/compressed'
forged_dir = 'dataset/forged/compressed'

X, y = [], []

# Load Authentic
auth_files = []
for ext in IMAGE_EXTENSIONS:
    auth_files.extend(glob.glob(os.path.join(auth_dir, ext)))
    auth_files.extend(glob.glob(os.path.join(auth_dir, ext.upper())))
auth_files = sorted(list(set(auth_files)))
print(f"Loading {len(auth_files)} Authentic samples...")
for f in auth_files:
    with Image.open(f) as img:
        ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
        X.append(np.array(ela, dtype=np.float32) / 255.0)
        y.append(0)

# Load Forged (including AI diffusion)
forged_files = []
for ext in IMAGE_EXTENSIONS:
    forged_files.extend(glob.glob(os.path.join(forged_dir, '**', ext), recursive=True))
    forged_files.extend(glob.glob(os.path.join(forged_dir, '**', ext.upper()), recursive=True))
forged_files = sorted(list(set(forged_files)))
print(f"Loading {len(forged_files)} Forged samples (including AI diffusion fakes)...")
for f in forged_files:
    with Image.open(f) as img:
        ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
        X.append(np.array(ela, dtype=np.float32) / 255.0)
        y.append(1)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

# Stratified Split: 70% Train, 15% Val, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Dataset Prepared: Total={len(X)} | Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

In [ ]:
# Step 4: Define 3 CNN Architectures
def build_basic_cnn():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_mobilenetv2():
    base = applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_resnet50():
    base = applications.ResNet50(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Step 5: Train All 3 Architectures and Evaluate Comparative Metrics
models_dict = {
    'Basic_CNN': build_basic_cnn(),
    'MobileNetV2': build_mobilenetv2(),
    'ResNet50': build_resnet50()
}

results = {}
for name, model in models_dict.items():
    print(f"\n{'='*20} Training {name} {'='*20}")
    t0 = time.time()
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=16, verbose=1)
    train_time = time.time() - t0
    
    # Test set inference
    t_inf = time.time()
    y_prob = model.predict(X_test)
    lat_ms = ((time.time() - t_inf) / len(X_test)) * 1000.0
    y_pred = (y_prob > 0.5).astype(int).flatten()
    
    results[name] = {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall': float(recall_score(y_test, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_test, y_pred, zero_division=0)),
        'latency_ms': float(lat_ms),
        'train_duration_s': float(train_time)
    }
    
    model.save(f"models/{name.lower()}.keras")
    print(f"Saved models/{name.lower()}.keras | Test Accuracy: {results[name]['accuracy']*100:.2f}%")

with open('models/evaluation_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n=== Comparative Architecture Results ===")
for k, v in results.items():
    print(f"{k:12s} | Acc: {v['accuracy']*100:.2f}% | Prec: {v['precision']*100:.2f}% | Rec: {v['recall']*100:.2f}% | F1: {v['f1_score']:.4f} | Latency: {v['latency_ms']:.2f}ms")